# Notebook2 - Baseline Anomaly Detection

用 rolling mean/std、rolling median/MAD 與 percentile threshold 建立可解釋的 baseline anomaly detection。

## Step 0 - 讀取 Notebook1 輸出的 features

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

def show_fig(fig):
    display(fig)
    plt.close(fig)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path("..").resolve()

DATA_SYNTHETIC = PROJECT_ROOT / "data" / "synthetic"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
features = pd.read_csv(DATA_PROCESSED / "features.csv", parse_dates=["timestamp"])
event_catalog = pd.read_csv(DATA_SYNTHETIC / "synthetic_event_catalog.csv", parse_dates=["start_time", "end_time"])
print(features.shape)
display(features[["timestamp", "port_id", "event_label", "traffic_total", "ucast_total", "errors_total", "discards_total"]].head())

## Step 1 - 定義 baseline 方法與 anomaly flags

每個監控維度同時計算 z-score、robust z-score 與 p95 exceed。

In [ ]:
flag_map = {
    "traffic_high": "traffic_total",
    "packets_high": "ucast_total",
    "errors_high": "errors_total",
    "discards_high": "discards_total",
    "broadcast_high": "broadcast_total",
    "multicast_high": "multicast_total",
    "unknown_high": "unknown_total",
}

result = features[["timestamp", "device_id", "port_id", "port_role", "event_label", "event_id"] + list(flag_map.values())].copy()
for flag, col in flag_map.items():
    mean = features[f"{col}_rolling_mean_1h"]
    std = features[f"{col}_rolling_std_1h"].replace(0, np.nan)
    median = features[f"{col}_rolling_median_1h"]
    mad = features[f"{col}_rolling_mad_1d"].replace(0, np.nan)
    p95 = features[f"{col}_rolling_p95_1d"]

    result[f"{col}_z_score"] = ((features[col] - mean) / std).replace([np.inf, -np.inf], 0).fillna(0)
    result[f"{col}_robust_z"] = (0.6745 * (features[col] - median) / mad).replace([np.inf, -np.inf], 0).fillna(0)
    result[f"{col}_percentile_exceed"] = features[col] > p95
    result[flag] = (
        (result[f"{col}_z_score"] > 3.0)
        | (result[f"{col}_robust_z"] > 4.0)
        | (result[f"{col}_percentile_exceed"] & (features[col] > 0))
    )

flag_cols = list(flag_map.keys())
result["baseline_alert_count"] = result[flag_cols].sum(axis=1)
result["any_baseline_anomaly"] = result["baseline_alert_count"] > 0
display(result.loc[result["any_baseline_anomaly"], ["timestamp", "port_id", "event_label", "baseline_alert_count"] + flag_cols].head(12))

## Step 2 - 視覺化：原始值 + baseline band + anomaly markers

In [ ]:
port_id = "port-id7427"
col = "traffic_total"
p = features[features["port_id"] == port_id].copy()
r = result[result["port_id"] == port_id].copy()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(p["timestamp"], p[col], label=col, linewidth=1)
ax.plot(p["timestamp"], p[f"{col}_rolling_mean_1h"], label="rolling_mean_1h", linewidth=1.2)
upper = p[f"{col}_rolling_mean_1h"] + 3 * p[f"{col}_rolling_std_1h"]
lower = (p[f"{col}_rolling_mean_1h"] - 3 * p[f"{col}_rolling_std_1h"]).clip(lower=0)
ax.fill_between(p["timestamp"], lower, upper, alpha=0.18, label="mean ± 3std")
anomalies = r[r["traffic_high"]]
ax.scatter(anomalies["timestamp"], anomalies[col], color="tab:red", s=18, label="traffic_high")
for _, event in event_catalog.iterrows():
    if event["port_id"] in [port_id, "MULTI"]:
        ax.axvspan(event["start_time"], event["end_time"], alpha=0.10, color="tab:red")
ax.set_title(f"{port_id} - Baseline anomaly detection")
ax.legend(loc="upper left")
ax.grid(alpha=0.25)
plt.tight_layout()
show_fig(fig)

## Step 3 - 視覺化：不同 flags 的時間分布

這對應 Grafana 的 anomaly flag time series 和 annotation。

In [ ]:
daily_flags = result.set_index("timestamp")[flag_cols].resample("6h").sum()
fig, ax = plt.subplots(figsize=(14, 5))
daily_flags.plot(ax=ax, linewidth=1.4)
ax.set_title("Baseline anomaly flags aggregated every 6 hours")
ax.set_ylabel("flag count")
ax.grid(alpha=0.25)
plt.tight_layout()
show_fig(fig)

event_recall = (
    result.groupby("event_label")
    .agg(rows=("timestamp", "size"), anomaly_rows=("any_baseline_anomaly", "sum"), avg_flags=("baseline_alert_count", "mean"))
    .assign(anomaly_rate=lambda x: x["anomaly_rows"] / x["rows"])
    .sort_values("anomaly_rate", ascending=False)
)
display(event_recall)

## Step 4 - 輸出 baseline_anomaly_flags.csv

In [ ]:
out_path = DATA_PROCESSED / "baseline_anomaly_flags.csv"
result.to_csv(out_path, index=False)
print(f"saved: {out_path}")